In [ ]:
import oracledb
import ssl
import pandas as pd

# === DADOS DE CONEXÃO === #
usuario = "BRUNOBRIANI_SCHEMA_8DADJ"
senha = "5RHMY4#9QDbPGMOBOLDXUXAK8WJ2WI"
dsn = "(description=(address=(protocol=tcps)(port=2484)(host=db.freesql.com))(connect_data=(service_name=23ai_34ui2)))"

# === Configuração SSL === #
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

try:
    # === Conexão === #
    connection = oracledb.connect(user=usuario, password=senha, dsn=dsn, ssl_context=ctx)
    print("Conectado com sucesso ao banco!")

    # === QUERYS SQL === #
    query01 = """
    SELECT 
        e.first_name, 
        e.last_name, 
        e.hire_date, 
        j.job_title, 
        d.department_name, 
        e.salary
    FROM HR.employees e
    LEFT JOIN HR.jobs j ON e.job_id = j.job_id
    LEFT JOIN HR.departments d ON e.department_id = d.department_id
    WHERE d.department_name IS NOT NULL 
      AND e.salary > 2800
    ORDER BY d.department_name, j.job_title, e.salary DESC
    """  

    query02 = """
    SELECT 
        e.employee_id,
        e.first_name,
        e.last_name,
        e.salary,
        e.hire_date, 
        d.department_name,
        l.city,
        c.country_name,
        r.region_name
    FROM HR.employees e
    LEFT JOIN HR.departments d ON e.department_id = d.department_id
    LEFT JOIN HR.locations l ON d.location_id = l.location_id
    LEFT JOIN HR.countries c ON l.country_id = c.country_id
    LEFT JOIN HR.regions r ON c.region_id = r.region_id
    WHERE r.region_name IS NOT NULL AND e.salary > 2800
    ORDER BY r.region_name, c.country_name, d.department_name, e.salary DESC
    """  

    # === Carrega os dados === #
    df1 = pd.read_sql(query01, connection)
    df2 = pd.read_sql(query02, connection)

    print("\n--- DADOS QUERY 1 ---")
    print(df1.head())
    
    print("\n--- DADOS QUERY 2 ---")
    print(df2.head())

    # === Exportar para a pasta criada (Dados) === #
    df1.to_csv("Dados/query_01.csv", index=False)
    df2.to_csv("Dados/query_02.csv", index=False)

    connection.close()
    print("\nConexão encerrada e arquivos CSV gerados na pasta Dados.")

except Exception as e:
    print(f"\nErro na execução: {e}")


In [ ]:
# === Análise Exploratória de Dados (EDA) QUERY_01 === #
import pandas as pd

# === 1. Carrega os Dados da Query 01 === #
df_1 = pd.read_csv("Dados/query_01.csv")

# === 2. Diagnóstico Dos Dados === #
print("=== DIAGNÓSTICO DOS DADOS PARA TODAS AS VARIÁVEIS ===")

# === Verifica nulos em todas as colunas === #
nulos = df_1.isnull().sum()
print(f"\nValores Nulos por coluna:\n{nulos[nulos > 0] if nulos.sum() > 0 else '0'}")

# === Verifica duplicados em todas as linhas === #
duplicados = df_1.duplicated().sum()
print(f"\nLinhas duplicadas encontradas no conjunto de dados total: {duplicados}")

# === Transformação de data para datetime === #
df_1['HIRE_DATE'] = pd.to_datetime(df_1['HIRE_DATE'])
df_1['HIRE_YEAR'] = df_1['HIRE_DATE'].dt.year

print("\n=== Base 100% verificada e limpa ===")


In [ ]:
import numpy as np

# === Análise Exploratória Básica === #
print("=== VISÃO GERAL DOS DADOS (QUERY 01) ===")
print(df_func.info())

# === Criar a coluna de Ano para a análise temporal === #
df_func['HIRE_DATE'] = pd.to_datetime(df_func['HIRE_DATE'])
df_func['HIRE_YEAR'] = df_func['HIRE_DATE'].dt.year

# === Lista de métricas estatísticas Query 01 === #
metricas = ['mean', 'median', 'min', 'max', 'std', 'var']

In [ ]:
# === Cálcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Departamento' === #

print("\n=== ESTATÍSTICA SALARIAL POR DEPARTAMENTO ===")
estatistica_depto = df_func.groupby('DEPARTMENT_NAME')['SALARY'].agg(metricas)
estatistica_depto.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']
from tabulate import tabulate

# === Arredondar todos os valores para 2 casas decimais === #
estatistica_depto = estatistica_depto.round(2)

# === Exibir tabela com o tabulate === #
print(tabulate(estatistica_depto.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))

In [ ]:
# === Cálcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Cargo' === #

print("\n=== ESTATÍSTICA SALARIAL POR CARGO ===")
estatistica_cargo = df_func.groupby('JOB_TITLE')['SALARY'].agg(metricas)
estatistica_cargo.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']

# Arredonda todos os valores para 2 casas decimais
estatistica_cargo = estatistica_cargo.round(2)

# Agora exiba com o tabulate (ou print)
print(tabulate(estatistica_cargo.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))

In [ ]:
# Calcula as métricas estatísticas e cria tabela para Distribuição de 'Salário X Ano de Contratação'

print("\n=== ESTATÍSTICA SALARIAL POR ANO DE CONTRATAÇÃO ===")
estatistica_ano = df_func.groupby('HIRE_YEAR')['SALARY'].agg(metricas)
estatistica_ano.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']

# === Arredonda os valores para 2 casas decimais === #
estatistica_ano = estatistica_ano.round(2)

# === Ordena corretamente usando sort_values === #
print(tabulate(estatistica_ano.sort_values(by='Média', ascending=False), 
               headers='keys', 
               tablefmt='pretty', 
               showindex=True))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# GRÁFICOS DE ANÁLISE SALARIAL POR DEPARTAMENTO #

# === Configuração de estilo === #
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Cria a figura com dois subplots ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# === Define a paleta de cores === #
paleta = sns.color_palette("viridis", n_colors=len(df_func['DEPARTMENT_NAME'].unique()))

# === GRÁFICO 1: Boxplot (Salário x Departamento) === #
sns.boxplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax1, 
    palette=paleta, 
    hue='DEPARTMENT_NAME', 
    legend=False,
    linewidth=1.2,
    fliersize=4
)
ax1.set_title('Distribuição Salarial por Departamento', fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Salário ($)')
ax1.set_ylabel('Departamento')

# === GRÁFICO 2: Média + Desvio Padrão (Estatística) === #
# = Design das Barras das médias com transparência = #
sns.barplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax2, 
    palette=paleta, 
    hue='DEPARTMENT_NAME', 
    alpha=0.4, 
    errorbar=None, 
    legend=False
)

# === Design das Linhas e pontos de Desvio Padrão === #
sns.pointplot(
    data=df_func, 
    x='SALARY', 
    y='DEPARTMENT_NAME', 
    ax=ax2, 
    color='black', 
    errorbar='sd', 
    join=False, 
    capsize=.15, 
    scale=0.6,                
    err_kws={'linewidth': 1.2} 
)

ax2.set_title('Média e Desvio Padrão', fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Salário Médio ($)')
ax2.set_ylabel('')

# === Ajuste final de layout === #
plt.tight_layout(pad=3.0)

# === Salva e exibe=== #
plt.savefig('analise_salarial_estatistica.png', dpi=300)
plt.show()

In [ ]:
# GRÁFICOS DE ANÁLISE SALARIAL POR CARGO #

# === Configuração de estilo === #
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Cria a figura  === #
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 12))

# === Define paleta de cores === #
paleta_cargo = sns.color_palette("viridis", n_colors=len(df_func['JOB_TITLE'].unique()))

# === GRÁFICO 1: Boxplot por Cargo === #
sns.boxplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax1, 
    palette=paleta_cargo, 
    hue='JOB_TITLE', 
    legend=False,
    linewidth=1.2,
    fliersize=4
)
ax1.set_title('Distribuição Salarial por Cargo', fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel('Salário ($)')
ax1.set_ylabel('Cargo')

# === GRÁFICO 2: Média + Desvio Padrão por Cargo === #
# Define design das Barras das médias
sns.barplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax2, 
    palette=paleta_cargo, 
    hue='JOB_TITLE', 
    alpha=0.4, 
    errorbar=None, 
    legend=False
)

# === Define design das Linhas de desvio padrão === #
sns.pointplot(
    data=df_func, 
    x='SALARY', 
    y='JOB_TITLE', 
    ax=ax2, 
    color='black', 
    errorbar='sd', 
    join=False, 
    capsize=.15, 
    scale=0.6,
    err_kws={'linewidth': 1.2}
)

ax2.set_title('Média e Desvio Padrão por Cargo', fontsize=18, fontweight='bold', pad=20)
ax2.set_xlabel('Salário Médio ($)')
ax2.set_ylabel('')

# === Ajuste final === #
plt.tight_layout(pad=3.0)

# === Salva e exibe === #
plt.savefig('analise_salarial_cargos.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# === Define a ordem dos anos do MAIOR para o MENOR === #
anos_invertidos = sorted(df_func['HIRE_YEAR'].unique(), reverse=True)

# === Configurações de estilo === #
sns.set_theme(style="white", context="paper")
plt.rcParams['font.family'] = 'sans-serif'
cor_base = "#88c999" 

corr, p_value = pearsonr(df_func['HIRE_YEAR'], df_func['SALARY'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# === GRÁFICO 1: Boxplot (Com ordem invertida) === #
sns.boxplot(data=df_func, x='HIRE_YEAR', y='SALARY', ax=ax1, 
            order=anos_invertidos,  # <--- ORDEM INVERTIDA AQUI
            color=cor_base, linewidth=0.8, fliersize=3,
            boxprops=dict(alpha=0.6, edgecolor='black', linewidth=0.8))
ax1.set_title('Distribuição Salarial (Ano mais recente p/ mais antigo)', fontsize=13, fontweight='bold', pad=15)
ax1.set_xlabel('Ano de Contratação')
ax1.set_ylabel('Salário (R$)')
sns.despine()

# === GRÁFICO 2: Tendência (Regressão invertida no eixo X) === #
sns.regplot(data=df_func, x='HIRE_YEAR', y='SALARY', ax=ax2, 
            scatter_kws={'alpha': 0.5, 'color': cor_base, 's': 30, 'edgecolor': 'white'}, 
            line_kws={'color': '#5a5a5a', 'linewidth': 1.5, 'linestyle': '--'})

# === Inverter o eixo X manualmente para o gráfico de dispersão === #
ax2.set_xlim(ax2.get_xlim()[::-1])

ax2.set_title(f'Tendência Temporal ($r = {corr:.2f}$)', fontsize=13, fontweight='bold', pad=15)
ax2.set_xlabel('Ano de Contratação (Decrescente)')
ax2.set_ylabel('')
sns.despine()

plt.tight_layout(pad=3.0)
plt.savefig('analise_temporal_invertida.png', dpi=300)
plt.show()

In [ ]:
import numpy as np

# ANÁLISE EXPLORATÓRIA DE DADOS (EDA) QUERY_02 #

# === Análise Exploratória Básica === #
print("=== VISÃO GERAL DOS DADOS (QUERY 02) ===")
print(df_2.info())

# === Criar a coluna de Ano para a análise temporal === #
df_2['HIRE_DATE'] = pd.to_datetime(df_2['HIRE_DATE'])
df_2['HIRE_YEAR'] = df_2['HIRE_DATE'].dt.year

# === Lista de métricas estatísticas Query 02 === #
metricas = ['mean', 'median', 'min', 'max', 'std', 'var']

In [ ]:
# === CRIA A ANÁLISE DAS MÉTRICAS E APRESENTA EM FORMA DE TABELA (Distribuição de 'SALÁRIO X REGIÃO' === #
print("\n=== ESTATÍSTICA SALARIAL POR REGIÃO E LOCALIZAÇÃO ===")

estatistica_regiao = df_2.groupby(['REGION_NAME', 'COUNTRY_NAME'])['SALARY'].agg(metricas)

estatistica_regiao.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']

# === Valores com duas casas decimais === #
estatistica_regiao = estatistica_regiao.round(2)

# === Substitui os 'nan' por 0 === #
estatistica_regiao[['Desvio Padrão', 'Variância']] = estatistica_regiao[['Desvio Padrão', 'Variância']].fillna(0)

from tabulate import tabulate

# === Exibe a tabela === #
print(tabulate(estatistica_regiao.sort_values(by='Média', ascending=False), 
                headers='keys', 
                tablefmt='pretty', 
                showindex=True))

In [ ]:
# === CRIA A TABELA E ANÁLISE DE FUNCIONÁRIOS POR REGIÃO E LOCALIZAÇÃO === #

print("\n=== NÚMERO DE FUNCIONÁRIOS POR REGIÃO E LOCALIZAÇÃO ===")

# === Realiza a contagem de linhas (funcionários) por grupo === #
contagem_regiao = df_2.groupby(['REGION_NAME', 'COUNTRY_NAME']).size().reset_index(name='Total de Funcionários')

# === Define o índice para manter a hierarquia visual (Região, País) === #
contagem_regiao.set_index(['REGION_NAME', 'COUNTRY_NAME'], inplace=True)

from tabulate import tabulate

# === Exibe tabela com o tabulate === #
print(tabulate(contagem_regiao.sort_values(by='Total de Funcionários', ascending=False), 
                headers='keys', 
                tablefmt='pretty', 
                showindex=True))

In [ ]:
# === CRIA A ANÁLISE E TABELA DE COMPARAÇÃO SALARIAL: Apenas Europa e Estados Unidos === #
print("\n=== COMPARAÇÃO SALARIAL: EUROPA VS. EUA ===")

# === Filtra apenas as regiões desejadas === #
filtro = df_2[df_2['REGION_NAME'].isin(['Europe', 'Americas'])]

# === Agrupa e aplica as métricas === #
estatistica_comparativa = filtro.groupby('REGION_NAME')['SALARY'].agg(metricas)

# === Mantém o padrão de nomes e arredondamento === #
estatistica_comparativa.columns = ['Média', 'Mediana', 'Mínimo', 'Máximo', 'Desvio Padrão', 'Variância']
estatistica_comparativa = estatistica_comparativa.round(2)

# === Trata possíveis 'nan' === #
estatistica_comparativa[['Desvio Padrão', 'Variância']] = estatistica_comparativa[['Desvio Padrão', 'Variância']].fillna(0)

from tabulate import tabulate

# === Exibe a tabela === #
print(tabulate(estatistica_comparativa.sort_values(by='Média', ascending=False), 
                headers='keys', 
                tablefmt='pretty', 
                showindex=True))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# === GRÁFICO: Distribuição Salarial por Localidade === #

# === Configuração de estilo (Limpo, sem grid) === #
sns.set_theme(style="white", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Aplica as abreviações para melhorar visualização do gráfico === #
df_2['COUNTRY_NAME'] = df_2['COUNTRY_NAME'].replace({
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK'
})

# === Cria a figura com dois subplots (horizontais) ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

# === Define a paleta de cores === #
paleta = sns.color_palette("viridis", n_colors=len(df_2['COUNTRY_NAME'].unique()))

# === GRÁFICO 1: Boxplot (Salário x Localidade) === #
sns.boxplot(
    data=df_2, 
    x='SALARY', 
    y='COUNTRY_NAME', 
    ax=ax1, 
    palette=paleta, 
    hue='COUNTRY_NAME', 
    legend=False,
    linewidth=1.2,
    fliersize=4
)
ax1.set_title('Distribuição Salarial por Localidade', fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel('Salário ($)')
ax1.set_ylabel('Localidade')
sns.despine(ax=ax1) # Remove bordas

# === GRÁFICO 2: Média + Desvio Padrão (Estatística) === #
# Barras das médias
sns.barplot(
    data=df_2, 
    x='SALARY', 
    y='COUNTRY_NAME', 
    ax=ax2, 
    palette=paleta, 
    hue='COUNTRY_NAME', 
    alpha=0.4, 
    errorbar=None, 
    legend=False
)

# Linhas de Desvio Padrão
sns.pointplot(
    data=df_2, 
    x='SALARY', 
    y='COUNTRY_NAME', 
    ax=ax2, 
    color='black', 
    errorbar='sd', 
    join=False, 
    capsize=.15, 
    scale=0.6,                   
    err_kws={'linewidth': 1.2} 
)

ax2.set_title('Média e Desvio Padrão', fontsize=18, fontweight='bold', pad=20)
ax2.set_xlabel('Salário Médio ($)')
ax2.set_ylabel('')
sns.despine(ax=ax2) # Remove bordas

# === Ajuste final de layout === #
plt.tight_layout(pad=4.0)

# === Salva e exibe === #
plt.savefig('estatistica_salarial_localidade.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# === GRÁFICO: Frequência de Funcionários por Localidade === #

# === Configuração de estilo: 'white' remove o grid cinza por padrão === #
sns.set_theme(style="white", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# === Renomeia localidades antes do gráfico === #
df_2['COUNTRY_NAME'] = df_2['COUNTRY_NAME'].replace({
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK'
})

# === Configuração da figura === #
plt.figure(figsize=(14, 8))

# === Dados e Paleta === #
contagem_data = df_2['COUNTRY_NAME'].value_counts()
paleta = sns.color_palette("viridis", n_colors=len(contagem_data))

# === GRÁFICO: Frequência de Funcionários por Localidade === #
ax = sns.barplot(
    x=contagem_data.index, 
    y=contagem_data.values, 
    palette=paleta, 
    hue=contagem_data.index, 
    legend=False
)

# === Personalização e Design === #
ax.set_title('Número de Funcionários por Localidade', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Localidade', fontsize=14)
ax.set_ylabel('Quantidade de Funcionários', fontsize=14)

# === Remove as linhas de grade === #
ax.grid(False) 

# === Remove as bordas superior e direita === #
sns.despine()

# === Rotaciona os nomes dos países === #
plt.xticks(rotation=45, ha='right', fontsize=12)

# === Adiciona os números acima de cada barra === #
for i in ax.containers:
    ax.bar_label(i, padding=3, fontsize=12)

# === Ajuste final === #
plt.tight_layout()
plt.savefig('frequencia_funcionarios_clean.png', dpi=300)
plt.show()

In [ ]:
# GRÁFICO: Heatmap de Comparação Salarial por Departamento e Localidade

# === Preparação dos dados para o Heatmap (usando DEPARTMENT_NAME) === #
# Cria uma tabela dinâmica (pivot) usando DEPARTAMENTO 
df_pivot = df_2.pivot_table(
    index='DEPARTMENT_NAME', 
    columns='COUNTRY_NAME', 
    values='SALARY', 
    aggfunc='mean'
)

# === Plotagem do Heatmap === #
plt.figure(figsize=(18, 12))

sns.heatmap(
    df_pivot, 
    annot=True,              
    fmt=".0f",               
    cmap="viridis",          
    linewidths=.5,           
    linecolor='white',       
    cbar_kws={'label': 'Média Salarialgi ($)'},
    annot_kws={"size": 10}   
)

# === Estilização do Gráfico === #
plt.title('Matriz de Comparação Salarial: Departamento vs. Localidade', fontsize=20, fontweight='bold', pad=30)
plt.xlabel('Localidade', fontsize=14, labelpad=15)
plt.ylabel('Departamento', fontsize=14, labelpad=15)

plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()
plt.savefig('matriz_comparacao_salarial_dept.png', dpi=300)
plt.show()